# 1. From a pile of text to a table

**The question this notebook answers: what is one row?**

That sounds like a formality. It is not. Many possible mistakes later in this course traces
back to a row meaning something different from what might assume — a claim about *people*
tested on a table of *messages*, an average over a unit nobody cares about, or a count that
silently double-counts.

So before any plot, you often have two jobs to consider doing:

1. **Get the data into rows that match the question.** In this notebook, that will mean a regular expression,
   because the raw material is a wall of text.
2. **Add what the question needs and the data does not have.** Enrichment. This is where
   the best analyses often are created

We do both on a showcase corpus first, where you can check your work against something
known, and then on your own chat.

The pipeline mechanics used from section 1.4 onward — what a `Pipeline` is, how to extend it,
why to type-hint a step's signature — live in
[`01.1-goad-toolkit-101.ipynb`](01.1-goad-toolkit-101.ipynb), on invented data. If you have
not run that one yet, do it first; here that machinery meets something real.

In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from goad_toolkit.datatransforms import Pipeline, TimeFeatures, TransformBase
from loguru import logger
from notebooktester import param

from wa_analyzer.data import load_showcase

## 1.1 The showcase: five years of Ubuntu IRC

Two public IRC channels, `#ubuntu-uk` and `#ubuntu-nl`, from 2013 to 2017. Real conversation
between real people, with the same awkward shape that real data sometimes has.

In [ ]:
irc = load_showcase("ubuntu_irc")
print(irc.shape)
irc.head()

Three columns and 3,371 rows. So one row is... what?

Look at the `text` column before assuming.

In [ ]:
day = irc.iloc[900]
print(f"{day.channel}  {day.created.date()}\n")
print(day.text[:400])

**One row is a whole day of one channel**, with every message of that day packed into a
single string.

That is not a useful unit for anything we want to ask. "How long are messages?" "Who talks
most?" "When is the channel busy?" — every one of those needs *one row per message*.

Nothing is broken here. The data was simply stored for a different purpose than ours. This
is normal, and noticing it is exactly the job of a data analyst.

## 1.2 Regular expressions: pulling structure out of text

A regular expression is a tiny language for describing the *shape* of text — "a bracket, two
digits, a colon, two digits, a bracket" rather than "the first eight characters." One pattern
replaces what would otherwise be a pile of `.split()` and `.startswith()` calls, which is why
regex shows up everywhere from `grep` to your editor's find-and-replace. It is also
legitimately hard to read back later — a pattern dense enough to be useful is often dense
enough that a different person, or you in a month, has to work it out character by character.
Both things are true at once: worth learning, worth writing carefully, and worth treating
with some suspicion once it is committed.

Every line in this corpus looks like this:

```
[00:08] <mapp> yay
```

Three things we want — a time, an author, a message — and some punctuation holding them
apart. Let us build the pattern against one line.

In [ ]:
line = "[00:00] <neuro> yay, it's officially pay day! (for me)"

pattern = re.compile(r"^\[(\d{2}):(\d{2})\]\s+<(\S+)>\s+(.*)$")
pattern.match(line).groups()  # ty: ignore[unresolved-attribute] -- known to match, it is the example line

Four groups back, in order. Reading the pattern piece by piece:

| piece | means |
|---|---|
| `^` | start of the line — so a `[` in the middle of a sentence cannot match |
| `\[` | a literal `[` starting the timestamp. Backslashed, because a bare `[` in regex has meaning |
| `(\d{2})` | **group**: A group () of digits `\d`, and exactly two `{2}` digits. `\d` is shorthand for `[0-9]` |
| `:` | a literal colon |
| `\]` | a literal `]` ending the timestamp, again backslashed to avoid the regex `]`|
| `\s+` | one or more `+` whitespace characters `\s`|
| `<(\S+)>` | angle brackets around a **group** `()` of one or more `+` *non*-whitespace characters `\S`|
| `(.*)` | **group**: any character `.` , zero or more times `*` — this should capture the message itself |
| `$` | end of the line |

The parentheses are what makes this useful. Without them the pattern only answers "does
this line look right?"; with them it hands back the four pieces.

One thing worth separating out: `\[` and `\]` above are *literal* brackets, backslashed
because a bare `[...]` already means something else in regex — a **character class**,
matching any single one of the characters inside it. `[0-9]` matches one digit, `[aeiou]`
matches one vowel, and `\d` in the table above is exactly `[0-9]` written shorter. Because
the timestamp's own brackets need to mean "a literal `[`", they get escaped; a bare `[` here
would instead try to open a character class and swallow the rest of the pattern trying to
close it. You will meet a character class again in the next section, spanning a range of raw
byte values rather than a handful of digits.

Two habits worth taking from this:

- **Anchor with `^` and `$`** unless you have a reason not to. An unanchored pattern will
  cheerfully find a match in the middle of something you did not mean.
- **`\S+` beats `.*` when you know there is no whitespace.** `.*` is greedy: it takes as much
  as it can and only gives back when forced. For a nickname that is wrong, and the bug it
  causes is subtle.

Now run it over everything. One row per line, per day, per channel.

In [ ]:
rows, unmatched = [], []
for day in irc.itertuples():
    for line in day.text.split("\n"):  # ty: ignore[unresolved-attribute]
        if not line.strip():
            continue
        m = pattern.match(line)
        if m:
            hh, mm, author, message = m.groups()
            rows.append((day.created, day.channel, int(hh), int(mm), author, message))  # ty: ignore[unresolved-attribute]
        else:
            unmatched.append((day.channel, line))  # ty: ignore[unresolved-attribute]

columns = pd.Index(["date", "channel", "hh", "mm", "author", "message"])
msgs = pd.DataFrame(rows, columns=columns)
print(f"{len(msgs):,} messages, {len(unmatched):,} lines did not match")
msgs.head()

## 1.3 The part everyone skips: what did *not* match?

615,683 rows is a satisfying number and it is the wrong thing to look at. The interesting
number is the other one.

A regex that matches most lines feels like success. But the lines it missed are not random —
they are the ones that are *shaped differently*, which is exactly where the surprises live.
Never accept a parse without looking at its leftovers.

In [ ]:
missed = pd.DataFrame(unmatched, columns=pd.Index(["channel", "line"]))
print(f"{len(missed):,} unmatched lines ({len(missed) / (len(msgs) + len(missed)):.2%})\n")

# Do not just eyeball a sample -- put the leftovers into named buckets.
missed["kind"] = "something else"
missed.loc[missed.line.str.match(r"^\[\d{2}:\d{2}\]\s+\*\s"), "kind"] = "action line"
missed.loc[missed.line.str.contains(r"[\x00-\x08\x0b-\x1f]"), "kind"] = "control characters"
print(missed.kind.value_counts().to_string(), "\n")

for kind, group in missed.groupby("kind"):
    print(f"{kind}:")
    print(f"   {group.line.iloc[0][:88]!r}\n")

Three different things are hiding in there.

**Action lines.** IRC has a `/me` command, and it logs differently — `* nick waves` rather
than `<nick> waves`. Same information, different shape. Our pattern was written for one and
not the other.

**Control characters.** Lines starting with `\x01` and `\x03` are what is left of IRC's
colour and formatting codes, partly stripped when this corpus was built. The message text
survived; the `[HH:MM] <nick>` prefix did not.

**The four "something else" lines**, like `'[21:19] <OerHeks>'`, are messages with nothing
after the nickname at all — not even a trailing space. Our pattern demands `\s+` between
`<nick>` and the message, so a genuinely empty message breaks it the same way an unanchored
pattern would break on something unexpected: correctly, on a case we did not think to write
for. Loosening that `\s+` to `\s*` would recover all four — but four lines out of 620,000 is
not worth the risk of a looser pattern matching something it should not elsewhere, so, like
the control characters, we name it, count it, and leave it alone.

The first is worth fixing — those are real messages by real people, and dropping them would
quietly bias anything we measure about who says what. The other two are damage or an edge
case too small to chase, and the honest move is to count them and move on rather than pretend
to recover what is not there.

In [ ]:
action = re.compile(r"^\[(\d{2}):(\d{2})\]\s+\*\s+(\S+)\s+(.*)$")

rows, still_missing = [], []
for day in irc.itertuples():
    for line in day.text.split("\n"):  # ty: ignore[unresolved-attribute]
        if not line.strip():
            continue
        m = pattern.match(line) or action.match(line)
        if m:
            hh, mm, author, message = m.groups()
            rows.append((day.created, day.channel, int(hh), int(mm), author, message,  # ty: ignore[unresolved-attribute]
                         bool(action.match(line))))
        else:
            still_missing.append(line)

columns = pd.Index(["date", "channel", "hh", "mm", "author", "message", "is_action"])
msgs = pd.DataFrame(rows, columns=columns)
coverage = len(msgs) / (len(msgs) + len(still_missing))
n_action = msgs.is_action.sum()
print(f"{len(msgs):,} messages, {coverage:.2%} of lines parsed")
print(f"{n_action:,} of those are action lines ({n_action / len(msgs):.2%} of all messages)")
print(f"of the {len(unmatched):,} lines the first pass missed, "
      f"{n_action / len(unmatched):.2%} turned out to be action lines")
print(f"{len(still_missing):,} lines still do not parse -- the control characters and "
      f"empty messages named above")

> "We recovered 99.94% of lines by also matching IRC's `/me` action syntax, which explains
> 96.7% of what the first pass missed; the remainder is corrupted control characters and a
> handful of empty messages."

This is a sentence that belongs in a report — it is the difference between
an analysis somebody can trust and one they cannot.

**That two-pass parse is worth keeping.** A new export, a new channel, next month's logs
would all need the identical regex rerun the same way — so before moving on, wrap the exact
two patterns above into one reusable step: check the column exists, do the parsing, hand back
a frame. That shape, one method called `transform`, is `TransformBase`'s contract;
[`01.1-goad-toolkit-101.ipynb`](01.1-goad-toolkit-101.ipynb) covers the mechanics if you have
not seen it yet.

In [ ]:
class ParseIRCLines(TransformBase):
    """Turn one-row-per-day IRC logs into one row per message.

    Combines the `<nick>` line pattern with the `/me` action-line fallback, and reports
    coverage through `loguru` instead of a manual print.
    """

    LINE = re.compile(r"^\[(\d{2}):(\d{2})\]\s+<(\S+)>\s+(.*)$")
    ACTION = re.compile(r"^\[(\d{2}):(\d{2})\]\s+\*\s+(\S+)\s+(.*)$")

    def transform(self, data: pd.DataFrame, text_column: str = "text") -> pd.DataFrame:
        rows, missing = [], 0
        for row in data.itertuples():
            for line in getattr(row, text_column).split("\n"):
                if not line.strip():
                    continue
                m = self.LINE.match(line) or self.ACTION.match(line)
                if m:
                    hh, mm, author, message = m.groups()
                    # itertuples() rows carry every column dynamically, so no stub can
                    # know `.created` and `.channel` exist ahead of time.
                    rows.append((row.created, row.channel, int(hh), int(mm), author,  # ty: ignore[unresolved-attribute]
                                 message, bool(self.ACTION.match(line))))
                else:
                    missing += 1

        columns = pd.Index(["date", "channel", "hh", "mm", "author", "message", "is_action"])
        parsed = pd.DataFrame(rows, columns=columns)
        coverage = len(parsed) / (len(parsed) + missing)
        logger.info(
            f"{self.name}: parsed {len(parsed):,} messages, {coverage:.2%} of lines "
            f"({missing:,} unparsed)"
        )
        return parsed

In [ ]:
by_hand = ParseIRCLines()(irc)
assert len(by_hand) == len(msgs)  # noqa: S101 -- the check *is* the point here: same logic, wrapped
print(f"{len(by_hand):,} messages -- identical to the two-pass parse above, now behind one call")

## 1.4 Pipelines: compile the steps

`ParseIRCLines` is one step. The enrichment that follows — calendar columns, a URL flag, a
question count, who a message is addressed to — is four more, and all five belong in one
`Pipeline` for the reason 101 lays out: a pipeline is a sequence you can print and rerun, not
a sequence you have to reconstruct by rereading cells in order.

**Some steps you never have to write.** `TimeFeatures` — calendar columns from a timestamp —
already ships in `goad_toolkit`:

In [ ]:
demo = Pipeline()
demo.add(TimeFeatures, column="date")
demo.apply(by_hand.head(3))[["date", "day_name", "isoweek", "year_week"]]

`TimeFeatures` came from the library, not this notebook, and `Pipeline.add` took the class
rather than an instance — the same rule the toy example in 101 made. Most of what you need to
extract will not ship, though: nobody packaged a transform for "does this IRC message address
someone by name." For that you write your own — same one-method contract, applied here
instead of on invented data.

In [ ]:
class RegexFeature(TransformBase):
    """Add a feature extracted from a text column with a regular expression.

    When `mode="extract"`, logs how many rows matched via `loguru` -- extraction is the
    one mode where "no match" is silent otherwise.
    """

    def transform(
        self,
        data: pd.DataFrame,
        column: str,
        pattern: str,
        feature: str,
        mode: str = "count",
    ) -> pd.DataFrame:
        text = data[column].fillna("")
        if mode == "count":
            data[feature] = text.str.count(pattern)
        elif mode == "has":
            data[feature] = text.str.contains(pattern, regex=True)
        elif mode == "extract":
            data[feature] = text.str.extract(pattern, expand=False)
            matched = data[feature].notna().sum()
            logger.info(
                f"{self.name}: extracted '{feature}' from {matched:,}/{len(data):,} rows "
                f"({matched / len(data):.1%}); {len(data) - matched:,} rows had no match"
            )
        else:
            raise ValueError(f"mode must be count/has/extract, got {mode!r}")
        return data

Two things about `RegexFeature` worth noticing, because both will bite you otherwise.

**The new column is called `feature`, not `name`.** `Pipeline.add(name=...)` already uses
`name` for the *step's* name, and the base class consumes it. Any transform that names an
output column has to call the parameter something else.

**Parameters are spelled out in the signature**, not swallowed by `**kwargs`. That is what
lets the base class check them, and it is what makes the step readable a month from now.

A third thing worth noticing, because it is the point of this section: **`mode="extract"`
reports its own coverage.** `count` and `has` fail loudly if something is wrong — a column of
all zeros or all `False` is visible the moment you look at it. `extract` fails silently: a row
with no match becomes `NaN`, which looks exactly like a row nobody checked. The manual
regex parse caught the same class of problem by hand, with a `print`. Here the step logs it
itself, via `loguru`, as a side effect of running — so the coverage number shows up every time
the pipeline runs, not just the one time you remembered to add a print for it.

Put it together: parse, then enrich, straight from the raw `irc` frame.

In [ ]:
full_pipeline = Pipeline()
full_pipeline.add(ParseIRCLines)
full_pipeline.add(TimeFeatures, column="date")
full_pipeline.add(RegexFeature, name="urls",
                   column="message", pattern=r"https?://\S+", feature="has_url", mode="has")
full_pipeline.add(RegexFeature, name="questions",
                   column="message", pattern=r"\?", feature="n_question", mode="count")
full_pipeline.add(RegexFeature, name="mentions",
                   column="message", pattern=r"^(\S+)[:,]\s", feature="addressed_to", mode="extract")

enriched = full_pipeline.apply(irc)
enriched.head()

`addressed_to` is the one worth pausing on. On IRC people answer each other by name —
`daftykins: try rebooting`. That single regex turns a flat list of messages into something
with a *reply structure*, which is a different kind of data entirely. Lesson 7 builds a
social graph out of exactly this column.

That is what enrichment means: not tidying, but adding a dimension the raw data did not
have.

And because the pipeline is an object, it can tell you what it did.

In [ ]:
print(full_pipeline)

> **Your turn, briefly.** Add one more step to `full_pipeline` above and re-run it. Some
> ideas: the number of capital letters, whether the message ends in a full stop, whether it
> contains a smiley, message length in words.
>
> Keep the ones you find interesting — lesson 5 uses features exactly like these to work out
> what distinguishes one person's writing from another's.

In [ ]:
# >>> add at least one more full_pipeline.add(...) step above this cell, then rerun >>>

your_turn = full_pipeline.apply(irc)
new_columns = set(your_turn.columns) - set(enriched.columns)

MIN_NEW_FEATURES = param(1, test=0)
assert len(new_columns) >= MIN_NEW_FEATURES, (  # noqa: S101 -- the check *is* the point here
    f"add at least {MIN_NEW_FEATURES} more full_pipeline.add(...) step above, so this produces "
    f"a column beyond {sorted(enriched.columns)}"
)
print(f"new column(s): {sorted(new_columns) or 'none yet -- add a step above'}")

### And `ParseIRCLines` does not belong in a notebook cell any more

It is yours, it works, and lesson 2 needs the same parsed frame. Those three facts together
mean it has outgrown this notebook. Copy the class into the next notebook and you have two
versions of one regex, which stay identical right up until the day one of them is fixed.

So it moves to `scripts/pipelines.py` — the same code, unchanged, plus a `build_irc_pipeline()`
function that assembles the five steps above. `scripts/` is installed along with this repo,
so importing it is an ordinary import:

In [ ]:
from scripts.pipelines import build_irc_pipeline

imported = build_irc_pipeline().apply(irc)
assert len(imported) == len(enriched)  # noqa: S101 -- the check *is* the point here
print(f"{len(imported):,} messages")

`build_irc_pipeline()` is a function rather than a ready-made `Pipeline` sitting in the module,
and that is deliberate. `pipeline["mentions"] = {"pattern": ...}` is goad's documented way to
retune a step — on a shared instance, one notebook doing that would silently change the
pipeline every other notebook imports. A factory hands each caller its own.

**Where code goes to live**, once it stops being a cell. Three places in this repo, and the
question each answers is "who else could possibly want this":

| where | what lives there | example |
|---|---|---|
| `goad_toolkit` | general-purpose, useful in any course | `TimeFeatures`, `RegexFeature` |
| `wa_analyzer` (`src/`) | this course's support code | `load_showcase` |
| `scripts/` | code *these lessons* derived and later ones reuse | `build_irc_pipeline` |

All three are installed, so all three are just imports. What separates them is scope, not
mechanism: `ParseIRCLines` parses one specific log format for one specific lesson, so it
would be wrong in `goad_toolkit` — and `scripts/` is exactly where "an example worth
keeping" belongs.

(One-off maintenance scripts — the ones that rebuild the showcase data or publish a dataset —
live in `tools/` and are *not* installed. Nobody imports those.)

That is the last thing this lesson does with the parse. From lesson 2 on, one import gets you
the frame you just built by hand.

One last thing about `RegexFeature`, now that yours works: **`goad_toolkit` ships it.**

`from goad_toolkit.datatransforms import RegexFeature` gets you the same three modes,
the same `feature`-not-`name` parameter, the same coverage log — you just independently
wrote what the library already had. That is the intended order. A transform you have
derived once is one you can read, debug and argue with; a transform you only ever
imported is a black box that happens to work.

From here on, import it. `ParseIRCLines` stays yours — it is specific to the shape of
these IRC logs, and nothing general enough to ship with `goad_toolkit`.

## 1.5 Finding bots not by content, but by patterns

The publishers of this corpus say they removed bots. Worth checking before you build on top
of it, because a bot is not a person and every per-author statistic will quietly include it.

We will find them **without reading any messages** — structure only. That habit generalises to all situations where the volumes are too big to "scroll trough the data"

In [ ]:
def author_profile(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n": len(g),
        "repeat_rate": 1 - g.message.nunique() / len(g),
        "active_days": g.date.nunique(),
        "busiest_day_share": g.groupby("date").size().max() / len(g),
        "median_length": g.message.str.len().median(),
    })


profiles = enriched.groupby("author").apply(author_profile, include_groups=False)
profiles = profiles[profiles.n >= 20]
print(f"{len(profiles):,} authors with at least 20 messages")

`repeat_rate` is the share of an author's messages that duplicate something they already
said. A script repeats itself constantly; a person, occasionally. So filter on it.

In [ ]:
suspects = profiles[profiles.repeat_rate > 0.4].sort_values("repeat_rate", ascending=False)
print(f"{len(suspects)} authors flagged: {', '.join(suspects.index)}")

Same numbers, now compared instead of listed. The eight highest `repeat_rate`s, with the 0.4
flag threshold marked — blue crossed it, grey did not. `lubotu3\`` is picked out in red,
because it is the one author here that actually is a bot. Watch where its bar lands.

In [ ]:
top = profiles.nlargest(8, "repeat_rate").reset_index()
palette = {
    author: "#c44e52" if author == "lubotu3`" else ("#4c72b0" if rate > 0.4 else "#cccccc")
    for author, rate in zip(top.author, top.repeat_rate)
}

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=top, x="repeat_rate", y="author", hue="author",
            palette=palette, legend=False, ax=ax)
ax.axvline(0.4, color="black", linestyle="--", linewidth=1)
ax.text(0.41, len(top) - 1, "flag threshold", va="center", fontsize=9)
ax.set_xlabel("repeat_rate")
ax.set_ylabel("")
ax.set_title("repeat_rate flags a human ritual and misses the actual bot")
fig.tight_layout()

The biggest of them is `brobostigon` — 9,161 messages over 1,797 days. Look at what they
repeated.

In [ ]:
brobo = enriched[enriched.author == "brobostigon"]
top = brobo.message.value_counts().head(3)
for text, count in top.items():
    days = brobo[brobo.message == text].date.nunique()
    print(f"{count:>5}x  on {days:>5} different days   {text!r}")

**"morning boys and girls."** 1,335 times, on 1,334 different days. Someone saying good
morning to their friends, every morning, for five years.

Now the bot.

In [ ]:
lubotu = enriched[enriched.author == "lubotu3`"]
for text, count in lubotu.message.value_counts().head(3).items():
    print(f"{count:>5}x  {text!r}")

And `lubotu3\`` is the actual bot — it pastes canned answers when someone says a keyword.
Its repeat rate is 0.386, *below* the threshold that caught `brobostigon`.

Repetition does not separate scripts from people, because people have rituals.

> That greeting returns in lesson 5, where how someone writes stops being noise and becomes
> the thing being measured.

## What counts as finished

Nothing above produced a headline finding. `brobostigon` crossed the repeat-rate threshold
this notebook picked; `lubotu3\`` sat just under it despite being the actual bot. That is a
judgment call sitting close to its own boundary, not a discovery — and saying so, instead of
moving the threshold until the story looks cleaner, is the norm for the rest of this course.

**A null result, honestly bounded, is a pass.** A report that says "no effect at this sample
size, and here is what would settle it" is a finished piece of work, not a lesser one than a
report with a significant p-value and no explanation attached to it. You will be asked to
defend every choice in your own analysis, assistant or no assistant — "I do not see it, and
here is why" is a defensible answer. "I kept adjusting the threshold until I found something"
is not, even if the number you end up reporting is real.
